<h1>Logistic Regression Model Build</h1>
<p>This script aims to train a logistic regression model.</p>

<h2>Import Libraries</h2>
<p>Only import required and used libraries in this script - removed unused libraries</p>

In [1]:
import os
import pandas as pd
import statsmodels.api as sm
import statsmodels.base.model as Model
import joblib

<h2>Read Data</h2>
<p>Reading in raw data</p>

In [2]:
cwd = os.getcwd().split('amts')[0] + 'amts'
theme = "machine_learning"

In [3]:
input_folder_path = rf"{cwd}/{theme}/data/outputs/01_lgbm_introduction"

In [4]:
input_file_path = rf"{input_folder_path}/train_input_df.csv"
print(rf"Input File (Prepped Input Train Data): {input_file_path}")
train_input_df = pd.read_csv(input_file_path)

Input File (Prepped Input Train Data): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/train_input_df.csv


In [5]:
input_file_path = rf"{input_folder_path}/test_input_df.csv"
print(rf"Input File (Prepped Input Test Data): {input_file_path}")
test_input_df = pd.read_csv(input_file_path)

Input File (Prepped Input Test Data): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/test_input_df.csv


<h2>Logistic Regression Model Build</h2>

In [6]:
print('\n---Start Logistic Regression Model Build---')


---Start Logistic Regression Model Build---


<h3>Create base function for model build</h3>

In [7]:
class LogisticRegressionModelBuilder:
    def __init__(self, input_train_data: pd.DataFrame, input_test_data: pd.DataFrame, independent_variables: list, target_variable: str
                 ) -> {Model, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame}:
        self.input_train_data = input_train_data
        self.input_test_data = input_test_data
        self.independent_variables = independent_variables
        self.target_variable = target_variable

    def run(self):
        self.X_train, self.y_train, self.X_test, self.y_test = self.split_data_by_independent_and_target_variables()
        self.X_train, self.X_test = self.add_constant()
        self.logistic_regression_model = self.train_logistic_regression_model()
        self.output_train_input_df, self.output_test_input_df = self.predict_target_variable()
        return self.logistic_regression_model, self.X_train, self.y_train, self.X_test, self.y_test, self.output_train_input_df, self.output_test_input_df

    def split_data_by_independent_and_target_variables(self): 
        X_train = self.input_train_data[self.independent_variables].copy()
        y_train = self.input_train_data[[self.target_variable]].copy()

        X_test = self.input_test_data[self.independent_variables].copy()
        y_test = self.input_test_data[[self.target_variable]].copy()
        return X_train, y_train, X_test, y_test
    
    def add_constant(self):
        X_train = sm.add_constant(self.X_train)
        X_test = sm.add_constant(self.X_test)
        return X_train, X_test

    def train_logistic_regression_model(self):
        logistic_regression_model = sm.Logit(self.y_train, self.X_train).fit()
        print(logistic_regression_model.summary())
        return logistic_regression_model

    def predict_target_variable(self):
        output_train_input_df = self.input_train_data.copy()
        output_test_input_df = self.input_test_data.copy()

        output_train_input_df['predicted_default_flag'] = self.logistic_regression_model.predict(self.X_train)
        output_test_input_df['predicted_default_flag'] = self.logistic_regression_model.predict(self.X_test)
        return output_train_input_df, output_test_input_df

<h3>Define target variable</h3>

In [8]:
target_variable_str = 'actual_default_flag'

<h3>Train different models with different independent variables combinations</h3>

In [9]:
model_id_to_independent_variables_mapping_dict = {
    'model_1': [
        'days_past_due_woe', 
        'balance_woe', 
        'credit_limit_woe', 
        'external_score_woe'
    ], 
    'model_2': [
        'days_past_due_woe', 
        'external_score_woe', 
        'credit_limit_woe'
    ], 
    'model_3': [
        'days_past_due_woe', 
        'external_score_woe', 
        'balance_woe'
    ], 
    'model_4': [
        'days_past_due_woe', 
        'external_score_woe'
    ], 
}

In [10]:
for model_id, independent_variables_list in model_id_to_independent_variables_mapping_dict.items():
    print(f'--{model_id.replace('_', ' ')}--')
    logistic_regression_model, X_train, y_train, X_test, y_test, output_train_input_df, output_test_input_df = LogisticRegressionModelBuilder(train_input_df, test_input_df, independent_variables_list, target_variable_str).run()
    print('\n')

--model 1--
Optimization terminated successfully.
         Current function value: 0.491629
         Iterations 6
                            Logit Regression Results                           
Dep. Variable:     actual_default_flag   No. Observations:                 3325
Model:                           Logit   Df Residuals:                     3320
Method:                            MLE   Df Model:                            4
Date:                 Wed, 04 Jun 2025   Pseudo R-squ.:                  0.2268
Time:                         17:19:37   Log-Likelihood:                -1634.7
converged:                        True   LL-Null:                       -2114.1
Covariance Type:             nonrobust   LLR p-value:                2.952e-206
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.7213      0.044    -16.346      0.000      -0.8

<h3>Compare average default flag between actual and predicted</h3>

In [11]:
print('--Average Actual vs Predicted Default Flag--')

--Average Actual vs Predicted Default Flag--


In [12]:
print('-Model 4: Train Data-')
print(output_train_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))

-Model 4: Train Data-
actual_default_flag       0.332331
predicted_default_flag    0.332331
dtype: float64


In [13]:
print('\n-Model 4: Test Data-')
print(output_test_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))


-Model 4: Test Data-
actual_default_flag       0.320000
predicted_default_flag    0.306632
dtype: float64


In [14]:
print('\n--End Logistic Regression Model Build--\n')


--End Logistic Regression Model Build--



<h2>Output</h2>
<p>Output model</p>

In [15]:
output_folder_path = rf"{cwd}/{theme}/data/outputs/01_lgbm_introduction"

In [16]:
output_file_path = rf"{output_folder_path}/logistic_regression_model.gz"
print(rf"Output File (Logistic Regression Model): {output_file_path}")
joblib.dump(logistic_regression_model, output_file_path)

Output File (Logistic Regression Model): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/logistic_regression_model.gz


['c:\\dev\\amts/machine_learning/data/outputs/01_lgbm_introduction/logistic_regression_model.gz']